# Graph Neural Net


In [ ]:
# packages
import os

import sys
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, accuracy_score, precision_score, f1_score, recall_score, confusion_matrix, silhouette_score
from sklearn.decomposition import PCA

from collections import deque

import torch

import seaborn as sns
import matplotlib.pyplot as plt

sys.path.append(str(Path().resolve().parent.parent / "src"))

# source files
from graph_neural_net.build_graph import build_graph
from graph_neural_net.model import GraphAE
from graph_neural_net.train import *
from graph_neural_net.utils import get_weight

RANDOM_SEED = 42
IMAGE_DIR = "../../results/images/GraphNeuralNet"
DATASET_DIR = "../../data/processed/unsupervised/"

torch.manual_seed(RANDOM_SEED)
np.random.default_rng(RANDOM_SEED)
os.makedirs(IMAGE_DIR, exist_ok=True)

In [ ]:
# step 1: load data
train_df = pd.read_csv(f'{DATASET_DIR}/train_data.csv')
val_df = pd.read_csv(f'{DATASET_DIR}/val_data.csv')
test_df = pd.read_csv(f'{DATASET_DIR}/test_data.csv')

y_val = val_df["fraud"]
y_test = test_df["fraud"]

In [ ]:
val_graph_df = pd.concat([train_df, val_df], ignore_index=True)
test_graph_df = pd.concat([val_graph_df, test_df], ignore_index=True)

In [ ]:
exclude_cols = {"customer", "merchant"}

exclude_cols.update([c for c in train_df.columns if c.startswith("age_")])
exclude_cols.update([c for c in train_df.columns if c.startswith("gender_")])
exclude_cols.update([c for c in train_df.columns if c.startswith("category_")])

feature_cols = [c for c in train_df.columns if c not in exclude_cols]

In [ ]:
train_graph = build_graph(train_df, feature_cols)
val_graph = build_graph(val_graph_df, feature_cols)
test_graph = build_graph(test_graph_df, feature_cols)

val_mask = torch.zeros(len(train_df) + len(val_df), dtype=torch.bool)
val_mask[len(train_df):] = True

test_mask = torch.zeros(len(train_df) + len(val_df) + len(test_df), dtype=torch.bool)
test_mask[len(train_df) + len(val_df):] = True

val_graph["transaction"].mask = val_mask
test_graph["transaction"].mask = test_mask

In [ ]:
weights = torch.tensor([get_weight(col) for col in feature_cols], dtype=torch.float32)
weights = weights / weights.mean()

In [ ]:
txn_input_dim = train_graph["transaction"].x.shape[1]

model = GraphAE(
    metadata=train_graph.metadata(),
    txn_input_dim=txn_input_dim,
    hidden_channels=64,
    dropout=0.2
)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-5)

In [ ]:
epochs = 90
warmup_epochs = 25


loss_window = deque(maxlen=5)
best_val_loss = float('inf')
best_model_state = None

losses = {"train": [], "val": []}

In [ ]:
for epoch in range(1, epochs + 1):
    train_loss = train_one_epoch(model, train_graph, optimizer, weights, noise=0.05)
    val_loss, scores, z_dict = evaluate(model, val_graph, weights)

    losses["train"].append(train_loss)
    losses["val"].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model = model.state_dict() 

    z_txn = z_dict["transaction"]
    z_masked = z_txn[val_graph["transaction"].mask]

    roc_auc = roc_auc_score(y_val, scores)
    pr_auc = average_precision_score(y_val, scores)

    print(
        f"Epoch {epoch}\n",
        f"Train Loss: {train_loss:.4f}\n",
        f"Val Loss: {val_loss:.4f}\n",
        f"ROC AUC: {roc_auc:.4f} ",
        f"PR AUC: {pr_auc:.4f} ",
    )

In [ ]:
model.load_state_dict(best_model)

plt.plot(range(epochs), losses["train"], label='Train Loss')
plt.plot(range(epochs), losses["val"], label='Validation Loss')
plt.ylabel('Loss/Error')
plt.xlabel('Epoch')
plt.savefig(f'{IMAGE_DIR}/loss_curve.png')
plt.legend()

model.eval()

In [ ]:
val_metrics, scores, z_dict = evaluate(model, val_graph, weights)

z_txn = z_dict["transaction"]
z_masked = z_txn[val_graph["transaction"].mask]

normal_idx = np.where(y_val == 0)[0]
fraud_idx = np.where(y_val == 1)[0]

normal_sample = np.random.choice(normal_idx, size=len(fraud_idx), replace=False)

idx = np.concatenate([normal_sample, fraud_idx])

pr_auc = average_precision_score(y_val, scores)
roc_auc = roc_auc_score(y_val, scores)
sl_score = silhouette_score(z_masked[idx], y_val[idx]) # Checks if fraud/normal embeddigns form distinct groups (closer to 1 more distinct)

print(f"Validation ROC AUC:   {roc_auc:.4f}")
print(f"Validation PR AUC:    {pr_auc:.4f}")
print(f"Silhoutte Score: {sl_score:.4f}")

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_val, scores)

plt.plot(precision, recall, label=f'Graph-NN (PR AUC = {pr_auc:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.savefig(f'{IMAGE_DIR}/pr_curve_validation.svg')
plt.show()

In [ ]:
beta = 2 # Makes F2 score, puts more emphasis on recall
fbeta_scores = (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall + 1e-8)

best_idx = fbeta_scores.argmax()
best_threshold = thresholds[best_idx]

torch.save({"model": model, "threshold": best_threshold, "base_df": train_df, "feature_cols": feature_cols , "weights": weights}, "../../results/models/graph_nn.pt" )

In [ ]:
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(z_masked)

plt.figure(figsize=(8, 6))
plt.scatter(emb_2d[y_val == 0, 0], emb_2d[y_val == 0, 1], alpha=0.5, label="Normal")
plt.scatter(emb_2d[y_val == 1, 0], emb_2d[y_val == 1, 1], alpha=0.5, label="Fraud")
plt.title("PCA of transaction embeddings")
plt.legend()
plt.savefig(f"{IMAGE_DIR}/pca_plot_validation.svg")
plt.show()

In [ ]:
test_metrics, scores, z_dict = evaluate(model, test_graph, weights)
scores = scores.numpy()

correct = 0
preds = []
for i, score in enumerate(scores):
    pred = 1 if score >= best_threshold else 0

    print(f'{i+1}.)\t Score: {score:.4f} \t predicted: {pred} \t actual: {int(y_test[i].item())}')

    if pred == int(y_test[i].item()):
        correct += 1

    preds.append(pred)

print(f"Correct: {correct} / {len(test_df)}")

In [ ]:
z_txn = z_dict["transaction"]
z_masked = z_txn[test_graph["transaction"].mask]

normal_idx = np.where(y_test == 0)[0]
fraud_idx = np.where(y_test == 1)[0]

normal_sample = np.random.choice(normal_idx, size=len(fraud_idx), replace=False)

idx = np.concatenate([normal_sample, fraud_idx])

accuracy = accuracy_score(y_test, preds)
precision = precision_score(y_test, preds, zero_division=0)
recall = recall_score(y_test, preds, zero_division=0)
f1 = f1_score(y_test, preds, zero_division=0)

pr_auc = average_precision_score(y_test, scores)
roc_auc = roc_auc_score(y_test, scores)
sil = silhouette_score(z_masked[idx], y_test[idx])

print(f"Test Accuracy:  {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall:    {recall:.4f}")
print(f"Test F1 Score:  {f1:.4f}")

print(f"Test ROC AUC:   {roc_auc:.4f}")
print(f"Test PR AUC:    {pr_auc:.4f}")
print(f"Test Silhoutte: {sil:.4f}")

In [ ]:
cf_matrix = confusion_matrix(y_test, preds)

group_names = ["True Negatives", "False Positives", "False Negatives", "True Positives"]

group_counts = ["{0:0.0f}".format(value) for value in cf_matrix.flatten()]

labels = [f"{v1}\n{v2}" for v1, v2 in zip(group_names,group_counts)]
labels = np.array(labels).reshape(2,2)

matrix = sns.heatmap(cf_matrix, annot=labels, fmt='', cmap="viridis", cbar=False)
matrix.set_xlabel("Predicted Label")
matrix.set_ylabel("True Label")
matrix.set_xticklabels(["Negative", "Positive"])
matrix.set_yticklabels(["Negative", "Positive"])
plt.title('Confusion Matrix')
plt.savefig(f'{IMAGE_DIR}/Confusion_Matrix.svg')
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, scores)

plt.plot(fpr, tpr, label=f"Graph-NN (AUC = {roc_auc:.4f})")
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.savefig(f'{IMAGE_DIR}/roc_curve_test.svg')
plt.show()

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, scores)

plt.plot(recall, precision, label=f"Graph-NN (AUC = {pr_auc:.4f})")
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.savefig(f'{IMAGE_DIR}/pr_curve_test.svg')
plt.show()

In [ ]:
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(z_masked)

plt.figure(figsize=(8, 6))
plt.scatter(emb_2d[y_test == 0, 0], emb_2d[y_test == 0, 1], alpha=0.5, label="Normal")
plt.scatter(emb_2d[y_test == 1, 0], emb_2d[y_test == 1, 1], alpha=0.5, label="Fraud")
plt.title("PCA of transaction embeddings")
plt.legend()
plt.savefig(f"{IMAGE_DIR}/pca_plot_test.svg")
plt.show()

In [ ]:
q99 = np.quantile(scores, 0.99)

plt.figure(figsize=(8, 5))

plt.hist(scores[y_test == 0], bins=50, alpha=0.5, label="Normal", density=True, range=(scores.min(), q99))
plt.hist(scores[y_test == 1], bins=50, alpha=0.5, label="Fraud", density=True, range=(scores.min(), q99))

plt.xlabel("Anomaly Score")
plt.ylabel("Density")
plt.title("Anomaly Score Distribution")
plt.axvline(best_threshold, color='red', linestyle='--', label='Threshold')
plt.legend()
plt.savefig(f"{IMAGE_DIR}/score_distribution.svg")
plt.show()